In [ ]:
import sys
import logging
from pathlib import Path

import torch

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_fold
from utils.mlp_utils import (
    set_seed,
    prepare_all_fold_tensors,
    run_nested_random_search,
    print_final_results,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

10:54:40 [INFO] Device: cuda


In [2]:
CFG = {
    "task": "hi",
    "dataset": "hiv",
    "fp_type": "ecfp4",
    "n_bits": 1024,

    "outer_folds": [1, 2, 3],

    "inner_k": 2,

    "random_state": GLOBAL_SEED,
}

In [3]:
SEARCH_SPACE = {
    "n_layers":     [1, 2, 3],
    "n_nodes":      [64, 128, 256, 512, 1024],
    "r":            [0.5, 0.7, 1.0],
    "activation":   ["relu", "leaky_relu", "gelu", "silu"],  
    "dropout":      [0.0, 0.2, 0.3, 0.5],
    "batchnorm":    [True, False],
    "init":         ["kaiming", "xavier"],

    "lr":           [1e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],
    "batch_size":   [64, 128, 256],
}

FIXED_HP = {
    "max_epochs": 100,
    "patience":   10,
    "grad_clip":  1.0,
}

N_ITER  = 150
N_SEEDS = 3

In [4]:
folds_data = {}

for fold_idx in CFG["outer_folds"]:
    train_df, test_df = load_fold(
        CFG["task"],
        CFG["dataset"],
        fold_idx,
    )

    folds_data[fold_idx] = {
        "train": train_df,
        "test": test_df,
    }

    n_train = len(train_df)
    n_test = len(test_df)

    n_train_pos = int(train_df["value"].sum())
    n_train_neg = n_train - n_train_pos

    n_test_pos = int(test_df["value"].sum())
    n_test_neg = n_test - n_test_pos

    logger.info(
        f"Fold {fold_idx} | "
        f"train={n_train} "
        f"(pos={n_train_pos}, neg={n_train_neg}, "
        f"pos_ratio={n_train_pos / n_train:.3f}) | "
        f"test={n_test} "
        f"(pos={n_test_pos}, neg={n_test_neg}, "
        f"pos_ratio={n_test_pos / n_test:.3f})"
    )

10:54:40 [INFO] Loaded hi/hiv fold 1: train=15696, test=7847
10:54:40 [INFO] Fold 1 | train=15696 (pos=599, neg=15097, pos_ratio=0.038) | test=7847 (pos=340, neg=7507, pos_ratio=0.043)
10:54:40 [INFO] Loaded hi/hiv fold 2: train=15695, test=7848
10:54:40 [INFO] Fold 2 | train=15695 (pos=504, neg=15191, pos_ratio=0.032) | test=7848 (pos=435, neg=7413, pos_ratio=0.055)


10:54:40 [INFO] Loaded hi/hiv fold 3: train=15695, test=7848
10:54:40 [INFO] Fold 3 | train=15695 (pos=775, neg=14920, pos_ratio=0.049) | test=7848 (pos=164, neg=7684, pos_ratio=0.021)


In [5]:
# SMILES → ECFP4 → NumPy arrays → PyTorch tensors

folds_tensors = prepare_all_fold_tensors(
    cfg=CFG,
    folds_data=folds_data,
    logger=logger,
)

10:54:40 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/hiv/ecfp4_train_1.npz
10:54:40 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/hiv/ecfp4_test_1.npz
10:54:40 [INFO] Fold 1 | X_train: (15696, 1024), X_test: (7847, 1024) | scale_features=False | pos_weight=25.204
10:54:40 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/hiv/ecfp4_train_2.npz
10:54:40 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/hiv/ecfp4_test_2.npz
10:54:40 [INFO] Fold 2 | X_train: (15695, 1024), X_test: (7848, 1024) | scale_features=False | pos_weight=30.141
10:54:40 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/hiv/ecfp4_train_3.npz
10:54:40 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/hiv/ecfp4_test_3.npz
10:54:40 [INFO] Fold 3 | X_train: (15695, 1024), X_test: (7848, 1024)

In [6]:
logger.info(f"Estimated time: ~{N_ITER * 6 * len(CFG['outer_folds']) / 60:.0f} min")

fold_results = run_nested_random_search(
    cfg=CFG,
    folds_tensors=folds_tensors,
    search_space=SEARCH_SPACE,
    fixed_hp=FIXED_HP,
    n_iter=N_ITER,
    n_seeds=N_SEEDS,
    device=device,
    seed=GLOBAL_SEED,
    logger=logger,
)

10:54:40 [INFO] Estimated time: ~45 min
10:54:40 [INFO] 
OUTER FOLD 1
10:54:53 [INFO]   [1/150] inner PR-AUC=0.4264 (13s) | L=2 N=512 r=0.7 dropout=0.3 lr=3e-03
10:55:13 [INFO]   [2/150] inner PR-AUC=0.4289 (20s) | L=3 N=128 r=1.0 dropout=0.2 lr=1e-03
10:55:20 [INFO]   [3/150] inner PR-AUC=0.4173 (7s) | L=1 N=1024 r=0.7 dropout=0.0 lr=5e-04
10:55:34 [INFO]   [4/150] inner PR-AUC=0.4167 (14s) | L=2 N=64 r=0.7 dropout=0.3 lr=3e-03
10:55:42 [INFO]   [5/150] inner PR-AUC=0.4163 (7s) | L=2 N=256 r=1.0 dropout=0.5 lr=3e-03
10:56:14 [INFO]   [6/150] inner PR-AUC=0.4115 (33s) | L=1 N=64 r=0.7 dropout=0.5 lr=1e-04
10:56:34 [INFO]   [7/150] inner PR-AUC=0.4294 (20s) | L=3 N=128 r=1.0 dropout=0.3 lr=1e-03
10:56:53 [INFO]   [8/150] inner PR-AUC=0.4093 (19s) | L=2 N=64 r=0.7 dropout=0.5 lr=5e-04
10:57:11 [INFO]   [9/150] inner PR-AUC=0.4139 (18s) | L=1 N=256 r=0.5 dropout=0.3 lr=1e-04
10:57:29 [INFO]   [10/150] inner PR-AUC=0.3830 (18s) | L=3 N=64 r=0.5 dropout=0.0 lr=5e-04
10:58:04 [INFO]   [11/15

In [7]:
import json
from pathlib import Path

out_dir = PROJECT_ROOT / "results" / CFG["task"] / CFG["dataset"] / f"mlp_{CFG['dataset']}_{CFG['task']}_{CFG['fp_type']}"
out_dir.mkdir(parents=True, exist_ok=True)

for res in fold_results:
    fold = res["fold"]

    with open(out_dir / f"params_fold_{fold}.json", "w") as f:
        json.dump({
            "fold": fold,
            "best_params": res["best_hp"],
            "inner_cv_score": res["inner_score"],
            "inner_train_loss_diagnostic": res["best_train_loss_diagnostic"],
            "test_metrics": res["test_metrics"],
            "seed_results": res["seed_results"],
        }, f, indent=2, default=str)

print(f"Saved JSON files in: {out_dir}")

Saved JSON files in: /home/f.capria/drug-discovery-lohi/results/hi/hiv/mlp_hiv_hi_ecfp4
